# Partie 4 — Statistique bivariée  
### Projet EDA — Détection de fraude bancaire

---

## 🎯 Objectifs

- Analyser les relations **QUANTI ↔ QUANTI** (corrélations entre composantes PCA et Amount)
- Analyser les relations **QUALI ↔ QUANTI** (Class → Amount, Class → Time)
- Construire une **heatmap de corrélation** pour identifier les variables discriminantes

---

## 📊 Récap des méthodes bivariées

| Relation | Exemple ici | Indicateurs | Graphiques |
|---------|-------------|-------------|------------|
| QUANTI ↔ QUANTI | V1 ↔ Amount | Covariance, Pearson, Spearman | Scatter plot |
| QUALI ↔ QUANTI | Class ↔ Amount | Moyennes par groupe | Boxplot, Violin |
| QUALI ↔ QUALI | (non applicable ici) | — | — |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

df = pd.read_csv("creditcard.csv")
print(f"Dataset chargé : {df.shape}")


---
# CAS 1 — QUANTI ↔ QUANTI  
**Exemple : V1 ↔ Amount, V14 ↔ Amount**

| Élément | Contenu |
|---------|---------|
| Variables | V1 (X), Amount (Y) |
| Indicateurs | Covariance, Corrélation de Pearson, Spearman |
| Graphique | Scatter plot |
| Objectif | Voir quelles composantes PCA sont liées au montant |


In [ ]:
# Corrélations avec Amount — top variables
corr_with_amount = df.drop(columns=["Class"]).corrwith(df["Amount"]).abs().sort_values(ascending=False)

print("Top 10 corrélations (|r|) avec Amount :")
print(corr_with_amount.head(10).round(4).to_string())


In [ ]:
# Scatter plots des composantes les plus corrélées à Amount
top_vars = corr_with_amount.drop("Time").head(4).index.tolist()

fig, axes = plt.subplots(1, len(top_vars), figsize=(14, 4))
for ax, var in zip(axes, top_vars):
    fraud   = df[df["Class"] == 1]
    normal  = df[df["Class"] == 0].sample(2000, random_state=42)
    
    ax.scatter(normal[var], normal["Amount"], alpha=0.2, s=8, color="#3B82F6", label="Normal")
    ax.scatter(fraud[var],  fraud["Amount"],  alpha=0.6, s=15, color="#EF4444", label="Fraude")
    
    r, p = stats.pearsonr(df[var], df["Amount"])
    ax.set_title(f"{var} vs Amount
r = {r:.3f}")
    ax.set_xlabel(var)
    ax.set_ylabel("Amount")
    ax.legend(fontsize=8, markerscale=2)

plt.suptitle("Scatter plots — composantes PCA vs Amount", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Corrélations Pearson et Spearman
x = df["V1"]
y = df["Amount"]
cov_xy  = np.cov(x, y, ddof=1)[0, 1]
r_p, p_p = stats.pearsonr(x, y)
r_s, p_s = stats.spearmanr(x, y)

print("V1 ↔ Amount")
print(f"  Covariance         : {cov_xy:.4f}")
print(f"  Pearson r          : {r_p:.4f}  (p = {p_p:.2e})")
print(f"  Spearman ρ         : {r_s:.4f}  (p = {p_s:.2e})")


---
# CAS 2 — QUALI ↔ QUANTI  
**Exemple : Class (fraude/normal) → Amount**

| Élément | Contenu |
|---------|---------|
| Variables | Class (quali), Amount (quanti) |
| Indicateurs | Moyennes, médianes par groupe |
| Graphiques | Boxplot, Violin plot, KDE |


In [ ]:
# Statistiques de Amount par Class
stats_by_class = df.groupby("Class")["Amount"].agg(
    count="count",
    moyenne="mean",
    mediane="median",
    std="std",
    min="min",
    q1=lambda x: x.quantile(0.25),
    q3=lambda x: x.quantile(0.75),
    max="max"
).round(2)
stats_by_class.index = ["Normal (0)", "Fraude (1)"]
print("Statistiques de Amount par classe :")
print(stats_by_class.to_string())


In [ ]:
# Boxplot et Violin plot — Amount par Class
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
df_plot = df.copy()
df_plot["Classe"] = df_plot["Class"].map({0: "Normal", 1: "Fraude"})
df_zoom = df_plot[df_plot["Amount"] < df_plot["Amount"].quantile(0.99)]

if sns:
    sns.boxplot(data=df_zoom, x="Classe", y="Amount", ax=axes[0],
                palette={"Normal": "#3B82F6", "Fraude": "#EF4444"})
    axes[0].set_title("Boxplot — Amount par classe")

    sns.violinplot(data=df_zoom, x="Classe", y="Amount", ax=axes[1],
                   palette={"Normal": "#3B82F6", "Fraude": "#EF4444"}, inner="quartile")
    axes[1].set_title("Violin plot — Amount par classe")

    sns.kdeplot(data=df_zoom, x="Amount", hue="Classe", ax=axes[2],
                fill=True, alpha=0.4, palette={"Normal": "#3B82F6", "Fraude": "#EF4444"})
    axes[2].set_title("KDE — Amount par classe")

plt.suptitle("Amount selon la classe (Normal vs Fraude)", fontsize=13)
plt.tight_layout()
plt.show()


---
## Analyse des composantes PCA par classe

Certaines composantes PCA ont été choisies pour maximiser la séparation des classes.  
On va identifier les composantes **les plus discriminantes**.


In [ ]:
# Différence de moyennes normalisée (Effect Size) par composante
v_cols = [f"V{i}" for i in range(1, 29)]

effect_sizes = []
for col in v_cols:
    fraud_vals  = df[df["Class"] == 1][col]
    normal_vals = df[df["Class"] == 0][col]
    
    diff_mean = abs(fraud_vals.mean() - normal_vals.mean())
    pooled_std = np.sqrt((fraud_vals.var() + normal_vals.var()) / 2)
    cohen_d = diff_mean / pooled_std if pooled_std > 0 else 0
    effect_sizes.append({"variable": col, "cohen_d": cohen_d,
                         "mean_fraud": fraud_vals.mean(), "mean_normal": normal_vals.mean()})

es_df = pd.DataFrame(effect_sizes).sort_values("cohen_d", ascending=False)
print("Top 10 composantes discriminantes (Cohen's d) :")
print(es_df.head(10)[["variable", "cohen_d", "mean_fraud", "mean_normal"]].round(3).to_string(index=False))


In [ ]:
# Visualisation : top 6 composantes discriminantes
top6 = es_df.head(6)["variable"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, var in zip(axes.flatten(), top6):
    if sns:
        df_plot2 = df.copy()
        df_plot2["Classe"] = df_plot2["Class"].map({0: "Normal", 1: "Fraude"})
        sns.kdeplot(data=df_plot2, x=var, hue="Classe", ax=ax,
                    fill=True, alpha=0.4, palette={"Normal": "#3B82F6", "Fraude": "#EF4444"})
        d = es_df[es_df["variable"] == var]["cohen_d"].values[0]
        ax.set_title(f"{var}  (d = {d:.2f})")

plt.suptitle("Top 6 composantes discriminantes — Normal vs Fraude", fontsize=13)
plt.tight_layout()
plt.show()
print("📌 Ces composantes seront les features les plus importantes pour la détection de fraude.")


---
## 📋 Récapitulatif — Partie 4

**Variables les plus discriminantes :**
- `V14`, `V10`, `V12`, `V4` montrent les plus grandes différences entre fraudes et transactions normales.
- `Amount` seul est peu discriminant — les fraudeurs adaptent leurs montants.

**Points clés :**
- Les composantes PCA issues de l'anonymisation contiennent **l'information discriminante**.
- La séparation normal/fraude est **visible graphiquement** sur plusieurs composantes.
- En partie 5, nous allons tester si ces différences sont **statistiquement significatives**.
